# 09C – REST API with FastAPI (Enterprise)

Build a production-ready REST API for bankruptcy prediction.

## Business Objective
Serve predictions from the trained model to external applications using FastAPI.

In [1]:
from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib
import uvicorn

MODEL_PATH='../models/production_bankruptcy_model.joblib'
model=joblib.load(MODEL_PATH)

app=FastAPI(title='Bankruptcy Risk Prediction API',version='1.0.0')


In [2]:
df=pd.read_csv('../data/datasets/american_bankruptcy_cleaned.csv')
target='status_label' if 'status_label' in df.columns else 'target'
features=[c for c in df.columns if c!=target]

PredictionRequest=type(
    'PredictionRequest',
    (BaseModel,),
    {'__annotations__':{c:float for c in features}}
)


In [3]:
@app.get('/')
def home():
    return {'message':'Bankruptcy Risk Prediction API','status':'running'}

@app.get('/health')
def health():
    return {'status':'healthy'}

@app.post('/predict')
def predict(request: PredictionRequest):
    X=pd.DataFrame([request.model_dump()])
    pred=int(model.predict(X)[0])
    prob=float(model.predict_proba(X)[0][1])
    return {
        'prediction':pred,
        'label':'Bankrupt' if pred else 'Healthy',
        'bankruptcy_probability':round(prob,4)
    }


In [4]:
# NOTE: uvicorn.run() is commented out so this notebook can be executed top-to-bottom
# without blocking on a live server. To actually serve the API, run this file as a
# script (e.g. `uvicorn 09C_REST_API_FastAPI_Enterprise:app --reload`) or uncomment below.
# if __name__=='__main__':
#     uvicorn.run(app,host='0.0.0.0',port=8000)
print('FastAPI app defined. Run with uvicorn as a script to serve it.')


FastAPI app defined. Run with uvicorn as a script to serve it.


## Endpoints

- GET /
- GET /health
- POST /predict

## Deliverables

- FastAPI service
- Health check endpoint
- Prediction endpoint
- JSON responses
- Ready for Docker deployment
